In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

data = pd.read_parquet('data/prepared_data.parquet')
print(data.shape)
print(f"Fraud procentas: {data['Is Fraud?'].mean()*100:.3f}%")

In [ ]:
data['date'] = pd.to_datetime(data[['Year', 'Month', 'Day']])
data_sorted = data.sort_values('date').reset_index(drop=True)

split_i = int(len(data_sorted) * 0.85)
train_val_data = data_sorted.iloc[:split_i].copy()
test_data = data_sorted.iloc[split_i:].copy()

train_val_data.drop(columns=['date', 'Year', 'Month', 'Day'], inplace=True)
test_data.drop(columns=['date', 'Year', 'Month', 'Day'], inplace=True)

print(f"Train: {len(train_val_data)} eil | fraud: {train_val_data['Is Fraud?'].mean()*100:.3f}%")
print(f"Test:  {len(test_data)} eil | fraud: {test_data['Is Fraud?'].mean()*100:.3f}%")

In [ ]:
X_tv = train_val_data.drop(columns=['Is Fraud?'])
y_tv = train_val_data['Is Fraud?']

X_test = test_data.drop(columns=['Is Fraud?'])
y_test = test_data['Is Fraud?']

X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=0.20,
    random_state=67,
    stratify=y_tv
)

print(f"Train: {len(X_train)} eil | fraud: {y_train.mean()*100:.3f}%")
print(f"Val: {len(X_val)} eil | fraud: {y_val.mean()*100:.3f}%")
print(f"Test: {len(X_test)} eil | fraud: {y_test.mean()*100:.3f}%")

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    precision_recall_curve, roc_curve
)
import matplotlib.pyplot as plt

def get_predictions(model, X, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)
    return y_proba, y_pred


def print_metrics(name, label, threshold, y, y_pred, y_proba):
    print("*************************************************************************")
    print(f"{name} | {label} | threshold={threshold:.3f}")
    print("*************************************************************************")
    print(classification_report(y, y_pred, target_names=['Ne-fraud', 'Fraud']))
    print(f"Precision: {precision_score(y, y_pred):.4f}")
    print(f"Recall: {recall_score(y, y_pred):.4f}")
    print(f"F1: {f1_score(y, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y, y_proba):.4f}")
    print(f"PR-AUC: {average_precision_score(y, y_proba):.4f}")
    print()
    print(f"Confusion matrix: {confusion_matrix(y, y_pred)}")


def plot_pr_curve(ax, y, y_pred, y_proba, threshold):
    prec, rec, _ = precision_recall_curve(y, y_proba)
    pr_auc = average_precision_score(y, y_proba)
    ax.plot(rec, prec, label=f'PR-AUC = {pr_auc:.4f}')
    ax.scatter(recall_score(y, y_pred), precision_score(y, y_pred),
               color='red', zorder=5, label=f'threshold={threshold:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall')
    ax.legend()
    ax.grid(True)


def plot_roc_curve(ax, y, y_proba):
    fpr, tpr, _ = roc_curve(y, y_proba)
    roc_auc = roc_auc_score(y, y_proba)
    ax.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate (Recall)')
    ax.set_title('ROC Curve')
    ax.legend()
    ax.grid(True)


def plot_curves(name, label, y, y_pred, y_proba, threshold):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_pr_curve(axes[0], y, y_pred, y_proba, threshold)
    plot_roc_curve(axes[1], y, y_proba)
    plt.suptitle(f'{name} | {label}')
    plt.tight_layout()
    plt.show()


def full_evaluate(name, model, X, y, threshold = 0.5, label=''):
    y_proba, y_pred = get_predictions(model, X, threshold)
    print_metrics(name, label, threshold, y, y_pred, y_proba)
    plot_curves(name, label, y, y_pred, y_proba, threshold)
    results = {
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'f1': f1_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba),
        'pr_auc': average_precision_score(y, y_proba),
    }
    return results

def cv_score(model, X, y, cv=5, label=''):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=67)
    scores = cross_validate(
        model, X, y,
        cv=skf,
        scoring=['roc_auc', 'average_precision'],
        n_jobs=-1
    )
    print()
    print(f"CV-{cv} | {label}]")
    print(f"ROC-AUC: {scores['test_roc_auc'].mean():.4f}")
    print(f"std ROC-AUC: {scores['test_roc_auc'].std():.4f}")
    print(f"PR-AUC: {scores['test_average_precision'].mean():.4f}")
    print(f"std PR-AUC: {scores['test_average_precision'].std():.4f}")
    return scores

def balance_data(X, y, fraud_target_ratio):
    ratio = fraud_target_ratio / (1 - fraud_target_ratio)
    target_for_under = ratio / 2

    n_fraud = (y == 1).sum()
    n_majority_target = int(n_fraud / target_for_under)
    under_ratio = n_majority_target / (y == 0).sum()
    under_ratio = min(under_ratio, 1.0)

    print(f"=== Prieš balansavimą ===")
    print(f"  Fraud: {n_fraud}, Non-fraud: {(y==0).sum()}")
    print(f"  Fraud %: {y.mean()*100:.1f}%")
    print(f"  under_ratio: {under_ratio:.4f}, target_for_under: {target_for_under:.4f}, ratio: {ratio:.4f}")

    under = RandomUnderSampler(sampling_strategy={0: n_majority_target, 1: n_fraud}, random_state=67)
    X_u, y_u = under.fit_resample(X, y)

    print(f"\n=== Po undersampling ===")
    print(f"  Fraud: {(y_u==1).sum()}, Non-fraud: {(y_u==0).sum()}")
    print(f"  Fraud %: {y_u.mean()*100:.1f}%")

    over = SMOTE(sampling_strategy=ratio, random_state=67)
    X_bal, y_bal = over.fit_resample(X_u, y_u)

    print(f"\n=== Po SMOTE ===")
    print(f"  Fraud: {(y_bal==1).sum()}, Non-fraud: {(y_bal==0).sum()}")
    print(f"  Fraud %: {y_bal.mean()*100:.1f}%")

    return X_bal, y_bal



In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

## eks 1: nebalansuota

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

xgb_no_bal = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=67,
    use_label_encoder=False,
    n_jobs=-1
)

cv_score(xgb_no_bal, X_train_scaled, y_train, cv=5, label='Nebalansuota')

xgb_no_bal.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    verbose=50
)

results_no_bal_val  = full_evaluate("XGB – Nebalansuota", xgb_no_bal, X_val_scaled,  y_val,  label='Validacija')
results_no_bal_test = full_evaluate("XGB – Nebalansuota", xgb_no_bal, X_test_scaled, y_test, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(xgb_no_bal, X_val_scaled)

results_no_bal = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_no_bal.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_no_bal)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

## eks 2: nebalansuota

In [ ]:
X_train_10, y_train_10 = balance_data(X_train_scaled, y_train, fraud_target_ratio=0.10)

xgb_10 = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=67,
    use_label_encoder=False,
    n_jobs=-1
)

cv_score(xgb_10, X_train_10, y_train_10, cv=5, label='10% fraud')

xgb_10.fit(
    X_train_10, y_train_10,
    eval_set=[(X_val_scaled, y_val)],
    verbose=50
)

results_10_val  = full_evaluate("XGB – 10% fraud", xgb_10, X_val_scaled,  y_val,  label='Validacija')
results_10_test = full_evaluate("XGB – 10% fraud", xgb_10, X_test_scaled, y_test, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(xgb_10, X_val_scaled)

results_10 = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_10.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_10)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

## eks 3: nebalansuota

In [ ]:
X_train_33, y_train_33 = balance_data(X_train_scaled, y_train, fraud_target_ratio=0.33)

xgb_33 = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=67,
    use_label_encoder=False,
    n_jobs=-1
)

cv_score(xgb_33, X_train_33, y_train_33, cv=5, label='33% fraud')

xgb_33.fit(
    X_train_33, y_train_33,
    eval_set=[(X_val_scaled, y_val)],
    verbose=50
)

results_33_val  = full_evaluate("XGB – 33% fraud", xgb_33, X_val_scaled,  y_val,  label='Validacija')
results_33_test = full_evaluate("XGB – 33% fraud", xgb_33, X_test_scaled, y_test, label='Testas')

In [ ]:
import numpy as np
import pandas as pd

y_proba_val, _ = get_predictions(xgb_33, X_val_scaled)

results_33 = []

thresholds = np.arange(0.001, 0.55, 0.05)

for t in thresholds:
    y_pred = (y_proba_val >= t).astype(int)
    results_33.append({
        'threshold': round(t, 3),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'pr_auc': average_precision_score(y_val, y_proba_val),
    })

df = pd.DataFrame(results_33)
print(df.to_string(index=False))

print(df.nlargest(3, 'f1').to_string(index=False))

---
## 6. XGBoost parametrai


| Parametras | Default | Ką daro |
|---|---|---|
| `n_estimators` | 100 | Kiek medžių |
| `max_depth` | 6 | Medžio gylis |
| `learning_rate` | 0.3 | Žingsnis |
| `subsample` | 1.0 | Kiek % eilučių naudojama kiekvienam medžiui |
| `colsample_bytree` | 1.0 | Kiek % požymių naudojama kiekvienam medžiui |
| `min_child_weight` | 1 | Min. svorių suma lape |
| `gamma` | 0 | Min. loss sumažėjimas splitui |
| `reg_alpha` | 0 | L1 regularizacija |
| `reg_lambda` | 1 | L2 regularizacija |

In [ ]:
X_train_best = X_train_10
y_train_best = y_train_10

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_child_weight': [1, 5, 10],
    'subsample': [0.7, 0.8, 1.0],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

base_xgb = xgb.XGBClassifier(
    n_estimators=200,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=67,
    use_label_encoder=False,
    n_jobs=-1
)

grid = GridSearchCV(
    base_xgb,
    param_grid,
    cv=skf,
    scoring='average_precision',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_best, y_train_best)

print("Geriausi parametrai:", grid.best_params_)
print(f"Geriausia CV PR-AUC: {grid.best_score_:.4f}")

In [ ]:
best_xgb = grid.best_estimator_

results_grid_val  = full_evaluate("XGB – GridSearch", best_xgb, X_val_scaled,  y_val,  label='Validacija')
results_grid_test = full_evaluate("XGB – GridSearch", best_xgb, X_test_scaled, y_test, label='Testas')

In [ ]:
import matplotlib.pyplot as plt

feature_names = X_train.columns.tolist()
importances = best_xgb.feature_importances_

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(fi_df['feature'][:15][::-1], fi_df['importance'][:15][::-1])
plt.title('XGBoost – Top 15 svarbiausi feature\'ai')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(fi_df.to_string(index=False))

In [ ]:
import joblib
import os
os.makedirs('models', exist_ok=True)

joblib.dump(best_xgb, 'models/xgb_best.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
